# 이전 1분기와 사건 이후 1분기 운임비 비교
**매주 컨테이너 1개(40ft)를 보내는 기업**이 사건 후 12주 동안 평시(사건 전 12주 평균)보다 운임비를 얼마나 더 냈는지 계산합니다.

- 입력: `financedata/해협별 운임비 변동 데이터_12주.xlsx`
- 출력: `이전 1분기와 사건 이후 1분기 운임비 비교.xlsx` (설명 / 위기별_요약 / 주차별_추가운임비)

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

SRC = DATA / "해협별 운임비 변동 데이터_12주.xlsx"
WEEKS = 12
CRISES = {"홍해": "KNEI (유럽향)", "호르무즈": "KMEI (중동향)"}

## 1. 타임라인 시트에서 위기별 주간 운임비 읽기

In [ ]:
raw = pd.read_excel(SRC, sheet_name="사건_반응_타임라인", header=None)


def to_date(v):
    if isinstance(v, (int, float)) and not pd.isna(v):   
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))
    return pd.to_datetime(v)


def read_block(route):
    h = raw.index[(raw[0] == "조사일") & (raw[1] == route)][0]
    ev_row = max(i for i in range(h) if raw.at[i, 0] == "사건일")
    ev = to_date(raw.at[ev_row, 1])
    rows = []
    for i in range(h + 1, len(raw)):
        if pd.isna(raw.at[i, 0]):
            break
        rows.append((to_date(raw.at[i, 0]), float(raw.at[i, 1])))
    return ev, pd.DataFrame(rows, columns=["조사일", "운임비_USD"])

## 2. 12주 격자로 정리하고 추가 운임비 계산
k주차 = 사건후 (7k−6)~7k일에 조사된 운임비. 연휴로 조사가 없는 주는 직전 조사값을 사용합니다.

In [ ]:
weekly, summary = [], []
for name, route in CRISES.items():
    ev, d = read_block(route)
    d["사건후_일수"] = (d["조사일"] - ev).dt.days
    pre = d[(d["사건후_일수"] >= -7 * WEEKS) & (d["사건후_일수"] < 0)]
    base = pre["운임비_USD"].mean()

    grid, prev = [], None
    for k in range(1, WEEKS + 1):
        obs = d[(d["사건후_일수"] >= 7 * k - 6) & (d["사건후_일수"] <= 7 * k)]
        if len(obs):
            prev = obs.iloc[0]
            grid.append({"주차": k, "조사일": prev["조사일"], "사건후_일수": prev["사건후_일수"],
                         "운임비_USD": prev["운임비_USD"], "비고": ""})
        else:
            grid.append({"주차": k, "조사일": pd.NaT, "사건후_일수": None, "운임비_USD": prev["운임비_USD"],
                         "비고": f"조사 없음 → 직전({prev['조사일']:%m-%d}) 값 사용"})
    g = pd.DataFrame(grid)
    g.insert(0, "위기", name)
    g["항로"] = route
    g["평시운임비_USD"] = round(base)
    g["추가운임비_USD"] = (g["운임비_USD"] - base).round(0)
    g["평시대비_배"] = (g["운임비_USD"] / base).round(2)
    g["누적추가운임비_USD"] = g["추가운임비_USD"].clip(lower=0).cumsum()
    weekly.append(g)

    total = g["추가운임비_USD"].clip(lower=0).sum()
    after = d[d["사건후_일수"] > 7 * WEEKS]
    summary.append({
        "위기": name, "항로": route, "사건일": ev.date(),
        "분기 끝(사건후 84일)": (ev + pd.Timedelta(days=84)).date(),
        "평시 기준 구간": f"{pre['조사일'].min():%Y-%m-%d} ~ {pre['조사일'].max():%Y-%m-%d} ({len(pre)}회 조사)",
        "평시 운임비(USD/FEU)": round(base),
        "사건 분기 평균 운임비(USD/FEU)": round(g["운임비_USD"].mean()),
        "최고 운임비(USD/FEU)": int(g["운임비_USD"].max()),
        "최고점(주차)": int(g.loc[g["운임비_USD"].idxmax(), "주차"]),
        "운임비 2배 도달(주차)": int(g.loc[g["평시대비_배"] >= 2, "주차"].min()),
        "분기 누적 추가 운임비(매주 1FEU, USD)": int(total),
        "평시였다면 낸 12주 총액(USD)": int(base * WEEKS),
        "추가 부담률(%)": round(total / (base * WEEKS) * 100, 1),
        "12주차 추가 운임비(USD/FEU)": int(g["추가운임비_USD"].iloc[-1]),
        "분기 직후 조사(확인용)": (f"{after['조사일'].iloc[0]:%Y-%m-%d} ${int(after['운임비_USD'].iloc[0]):,} "
                               f"(평시의 {after['운임비_USD'].iloc[0] / base:.1f}배)") if len(after) else "",
    })
weekly = pd.concat(weekly, ignore_index=True)
summary = pd.DataFrame(summary)
summary.T

## 3. 저장

In [ ]:
notes = pd.DataFrame([
    ("목적", "해협 위기 발생 후 한 분기(12주) 동안 기업이 평소보다 더 낸 운임비를 금액(USD)으로 계산"),
    ("자료", "financedata/해협별 운임비 변동 데이터_12주.xlsx [사건_반응_타임라인] (KCCI 주간, 부산발 40ft 1개당 USD)"),
    ("기간 기준", "사건 다음 날부터 84일(12주) = 한 분기. 기업의 분기 실적·운임 계약 단위와 맞춤"),
    ("주차", "k주차 = 사건후 (7k−6)~7k일에 조사된 운임비. 조사가 없는 주(연휴)는 직전 조사값 사용"),
    ("평시 운임비", "사건 전 12주(한 분기) 동안 조사된 운임비 평균 → '사건 전 한 분기 vs 사건 후 한 분기' 비교"),
    ("추가 운임비", "그 주 운임비 − 평시 운임비"),
    ("분기 누적 추가 운임비", "매주 컨테이너 1개를 보내는 기업이 12주 동안 더 낸 금액(추가분이 음수인 주는 0)"),
    ("추가 부담률", "누적 추가 운임비 ÷ (평시 운임비 × 12주)"),
    ("한계", "분기 이후에도 운임비가 평시의 3배 이상이라 피해는 계속됨 → 이 금액은 최소 피해액"),
], columns=["항목", "내용"])

weekly_out = weekly[["위기", "주차", "조사일", "사건후_일수", "운임비_USD", "비고", "항로", "평시운임비_USD",
                     "추가운임비_USD", "평시대비_배", "누적추가운임비_USD"]]
out = SAVE / "이전 1분기와 사건 이후 1분기 운임비 비교.xlsx"
with pd.ExcelWriter(out) as w:
    notes.to_excel(w, sheet_name="설명", index=False)
    summary.to_excel(w, sheet_name="위기별_요약", index=False)
    weekly_out.to_excel(w, sheet_name="주차별_추가운임비", index=False)
print("저장:", out)